In [ ]:
!pip install ultralytics --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.4 MB/s eta 0:00:00


In [ ]:
# Run this before the extractor. Trims each video to just the rally.
# Adjust start/end seconds per video after watching them.

from moviepy.editor import VideoFileClip

trims = [
    ("/content/drive/MyDrive/compress train vids/clip_01_19_56_to_01_20_17.mp4", 1.0, 19.0),   # start_sec, end_sec

]

for path, start, end in trims:
    clip = VideoFileClip(path).subclip(start, end)
    out = path.replace(".mp4", "_trimmed.mp4")
    clip.write_videofile(out, codec="libx264", audio=False)
    clip.close()
    print(f"  {path} → {end - start:.1f}s")

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



Moviepy - Building video /content/drive/MyDrive/compress train vids/clip_01_19_56_to_01_20_17_trimmed.mp4.
Moviepy - Writing video /content/drive/MyDrive/compress train vids/clip_01_19_56_to_01_20_17_trimmed.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/compress train vids/clip_01_19_56_to_01_20_17_trimmed.mp4
  /content/drive/MyDrive/compress train vids/clip_01_19_56_to_01_20_17.mp4 → 18.0s


In [ ]:
"""
Volleyball Frame Extractor + YOLO26x Person Detection (Multi-Video)
====================================================================
Run this in Google Colab with a T4 GPU runtime.

Processes MULTIPLE rally videos into a single unified output folder
with one detections.json and one frames/ directory. Each frame is
prefixed with the video ID to avoid naming collisions.

Workflow:
  1. Upload all rally videos to Colab
  2. List them in VIDEO_PATHS below
  3. Run all cells — produces one combined output
  4. Download and open with the HTML labeler for one unified session

Output structure:
  volleyball_output/
    frames/
      v0_frame_000030.jpg    <- video 0, frame 30
      v0_frame_000035.jpg
      ...
      v1_frame_000030.jpg    <- video 1, frame 30
      ...
    detections.json          <- all videos, all frames, all detections
"""

# ============================================================
# CELL 1 — Install dependencies (run once, then restart runtime)
# ============================================================
# !pip install ultralytics --quiet

# ============================================================
# CELL 2 — Configuration
# ============================================================
import os

# --- USER CONFIG -----------------------------------------------------------
VIDEO_PATHS = [
    "/content/clip_01_19_56_to_01_20_17_trimmed.mp4",
    "/content/clip_01_37_16_to_01_37_32_clipped.mp4",
    "/content/clip_01_44_20_to_01_44_57_clipped.mp4",
    "/content/clip_01_44_51_to_01_45_04.mp4",
    "/content/clip_42_42_to_43_06_clipped.MP4",
]

OUTPUT_DIR = "/content/volleyball_output"
FRAME_STEP = 5           # Extract every Nth frame (~12 fps at 60fps source)
SKIP_START = 25           # Skip first N frames
SKIP_END = 120            # Skip last N frames
MODEL_NAME = "yolo26x.pt" # Extra-large model for distant/small players
CONFIDENCE = 0.25         # Lower threshold — keep more, prune in labeler
IMG_SIZE = 1280           # Higher res for small-player detection
PERSON_CLASS_ID = 0       # COCO class 0 = person
# ---------------------------------------------------------------------------

os.makedirs(os.path.join(OUTPUT_DIR, "frames"), exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Videos to process: {len(VIDEO_PATHS)}")
print(f"Frame step: every {FRAME_STEP} frames")
print(f"Skip: first {SKIP_START}, last {SKIP_END} frames per video")

# ============================================================
# CELL 3 — Load YOLO26x with CUDA + FP16
# ============================================================
import torch
from ultralytics import YOLO

assert torch.cuda.is_available(), "No GPU detected — switch to a T4 runtime!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"CUDA version: {torch.version.cuda}")

model = YOLO(MODEL_NAME)
model.to("cuda")

# Warm-up run
dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device="cuda", dtype=torch.float16)
_ = model.predict(source=dummy, imgsz=IMG_SIZE, half=True, verbose=False)
print("Model loaded and warmed up on CUDA with FP16.")

# ============================================================
# CELL 4 — Extract frames + run detection across ALL videos
# ============================================================
import cv2
import json
import time

detections_data = {
    "frame_step": FRAME_STEP,
    "skip_start": SKIP_START,
    "skip_end": SKIP_END,
    "model": MODEL_NAME,
    "img_size": IMG_SIZE,
    "confidence_threshold": CONFIDENCE,
    "classes": {
        "dig": "Defensive pass / receive (forearm contact)",
        "set": "Overhead ball positioning for attack",
        "attack": "Offensive strike over the net",
        "block": "Defensive jump at the net to intercept",
        "serve": "Rally-initiating action",
        "idle": "No notable action",
        "ignore": "Not a player (coach, spectator, etc.)"
    },
    "videos": [],
    "frames": []
}

total_processed = 0
total_detections = 0
start_time = time.time()

for vid_idx, video_path in enumerate(VIDEO_PATHS):
    print(f"\n{'='*60}")
    print(f"VIDEO {vid_idx}: {os.path.basename(video_path)}")
    print(f"{'='*60}")

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"  Cannot open: {video_path} — skipping")
        continue

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    vid_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    vid_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = total_frames / fps if fps > 0 else 0
    end_frame = total_frames - SKIP_END
    extractable = max(0, end_frame - SKIP_START) // FRAME_STEP

    # Store video metadata
    detections_data["videos"].append({
        "video_id": vid_idx,
        "video_path": video_path,
        "filename": os.path.basename(video_path),
        "total_frames": total_frames,
        "fps": fps,
        "width": vid_w,
        "height": vid_h,
        "duration_sec": round(duration, 2),
    })

    print(f"  {vid_w}x{vid_h}, {fps:.1f} fps, {duration:.1f}s, "
          f"{total_frames} frames -> ~{extractable} to extract")

    frame_idx = 0
    vid_processed = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if (frame_idx >= SKIP_START and
            frame_idx <= end_frame and
            frame_idx % FRAME_STEP == 0):

            # Filename includes video ID prefix
            frame_filename = f"v{vid_idx}_frame_{frame_idx:06d}.jpg"
            frame_path = os.path.join(OUTPUT_DIR, "frames", frame_filename)
            cv2.imwrite(frame_path, frame, [cv2.IMWRITE_JPEG_QUALITY, 95])

            # YOLO26x inference
            results = model.predict(
                source=frame,
                imgsz=IMG_SIZE,
                half=True,
                conf=CONFIDENCE,
                classes=[PERSON_CLASS_ID],
                verbose=False,
                device="cuda",
            )

            # Parse detections
            frame_detections = []
            if results and results[0].boxes is not None:
                boxes = results[0].boxes
                for i in range(len(boxes)):
                    x1, y1, x2, y2 = boxes.xyxy[i].cpu().numpy().tolist()
                    conf = float(boxes.conf[i].cpu().numpy())
                    frame_detections.append({
                        "id": i,
                        "bbox": {
                            "x1": round(x1, 1),
                            "y1": round(y1, 1),
                            "x2": round(x2, 1),
                            "y2": round(y2, 1)
                        },
                        "confidence": round(conf, 3),
                        "label": "idle"
                    })

            detections_data["frames"].append({
                "video_id": vid_idx,
                "video_filename": os.path.basename(video_path),
                "frame_index": frame_idx,
                "timestamp": round(frame_idx / fps, 3),
                "filename": frame_filename,
                "width": frame.shape[1],
                "height": frame.shape[0],
                "detections": frame_detections
            })

            vid_processed += 1
            total_detections += len(frame_detections)

        frame_idx += 1

    cap.release()
    total_processed += vid_processed
    vid_dets = sum(len(f['detections']) for f in detections_data['frames']
                   if f['video_id'] == vid_idx)
    print(f"  Extracted {vid_processed} frames, {vid_dets} detections")

elapsed = time.time() - start_time
print(f"\n{'='*60}")
print(f"ALL DONE — {len(VIDEO_PATHS)} videos")
print(f"  Total frames extracted: {total_processed}")
print(f"  Total detections: {total_detections} "
      f"({total_detections/total_processed:.1f} avg/frame)")
print(f"  Time: {elapsed:.1f}s ({total_processed/elapsed:.1f} fps)")

# ============================================================
# CELL 5 — Save JSON
# ============================================================
json_path = os.path.join(OUTPUT_DIR, "detections.json")
with open(json_path, "w") as f:
    json.dump(detections_data, f, indent=2)

print(f"\nSaved: {json_path}")
print(f"Frames dir: {OUTPUT_DIR}/frames/")

# Per-video summary
print("\nPer-video breakdown:")
for vid_info in detections_data["videos"]:
    vid_id = vid_info["video_id"]
    vid_frames = [f for f in detections_data["frames"] if f["video_id"] == vid_id]
    vid_dets = sum(len(f["detections"]) for f in vid_frames)
    print(f"  v{vid_id} ({vid_info['filename']}): "
          f"{len(vid_frames)} frames, {vid_dets} detections, "
          f"{vid_info['duration_sec']}s")

# ============================================================
# CELL 6 — Zip & download
# ============================================================
import shutil

zip_path = shutil.make_archive(
    "/content/volleyball_output",
    "zip",
    root_dir="/content",
    base_dir="volleyball_output"
)
print(f"\nZipped: {zip_path}")
print("Download this zip, extract it, and open the HTML labeler.")
print("All 5 videos will appear as one continuous labeling session.")

Output directory: /content/volleyball_output
Videos to process: 5
Frame step: every 5 frames
Skip: first 25, last 120 frames per video
GPU: Tesla T4
CUDA version: 12.8
Model loaded and warmed up on CUDA with FP16.

VIDEO 0: clip_01_19_56_to_01_20_17_trimmed.mp4
  1920x1080, 62.1 fps, 18.0s, 1118 frames -> ~194 to extract
  Extracted 195 frames, 4822 detections

VIDEO 1: clip_01_37_16_to_01_37_32_clipped.mp4
  1920x1080, 62.0 fps, 15.7s, 975 frames -> ~166 to extract
  Extracted 167 frames, 3674 detections

VIDEO 2: clip_01_44_20_to_01_44_57_clipped.mp4
  1920x1080, 60.0 fps, 36.5s, 2188 frames -> ~408 to extract
  Extracted 409 frames, 9690 detections

VIDEO 3: clip_01_44_51_to_01_45_04.mp4
  1920x1080, 62.1 fps, 12.9s, 803 frames -> ~131 to extract
  Extracted 132 frames, 3199 detections

VIDEO 4: clip_42_42_to_43_06_clipped.MP4
  1920x1080, 62.0 fps, 20.9s, 1297 frames -> ~230 to extract
  Extracted 231 frames, 5597 detections

ALL DONE — 5 videos
  Total frames extracted: 1134
  Tot

In [ ]:
from google.colab import files

files.download('volleyball_output.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
"""
Volleyball Clip Extractor
===================================================================

Takes your labeled detections.json + original practice video and produces
player-centered video clips organized by action class, ready for VideoMAE
fine-tuning (potentially).

Each clip is a 2-second window (16 frames sampled uniformly) centered on
the labeled frame, cropped around the player's bounding box.

Output structure:
  clips/
    train/
      dig/       clip_f000120_d3.mp4
      set/       clip_f000450_d1.mp4
      attack/    clip_f000780_d5.mp4
      block/     clip_f001020_d2.mp4
      serve/     clip_f001500_d0.mp4
      idle/      clip_f000200_d4.mp4
    val/
      dig/  ...
      ...
    clip_manifest.json   <- metadata for every clip
"""

!pip install opencv-python-headless numpy

# ============================================================
# CONFIG
# ============================================================
import os
import json
import random
import cv2
import numpy as np


# --- USER CONFIG ---
VIDEO_PATHS = [
    "clip_01_19_56_to_01_20_17_trimmed.mp4",
    "clip_01_37_16_to_01_37_32_clipped.mp4"
    "clip_01_44_20_to_01_44_57_clipped.mp4",
    "clip_01_44_51_to_01_45_04.mp4",
    "clip_42_42_to_43_06_clipped.MP4",
]
LABELED_JSON = "detections_labeled.json"
OUTPUT_DIR = "clips"

NUM_FRAMES = 16          # VideoMAE expects 16 frames
CLIP_WINDOW_SEC = 2.0    # Total temporal window in seconds
CROP_PAD_RATIO = 1.5     # Padding around bbox (1.5 = 50% extra on each side)
CROP_SIZE = 224           # Final square crop size for VideoMAE
VAL_SPLIT = 0.15          # 15% held out for validation
SEED = 42
# -------------------

random.seed(SEED)
np.random.seed(SEED)

# ============================================================
# LOAD DATA
# ============================================================
with open(LABELED_JSON, 'r') as f:
    data = json.load(f)

# Build video lookup from the JSON metadata or from VIDEO_PATHS
# The JSON contains video_id per frame; map video_id → file path
video_map = {}  # video_id -> {path, fps, width, height, total_frames}

if 'videos' in data:
    # Use metadata from the JSON (written by multi-video extractor)
    for v in data['videos']:
        vid_id = v['video_id']
        video_map[vid_id] = {
            'path': v['video_path'],
            'fps': v['fps'],
            'width': v['width'],
            'height': v['height'],
            'total_frames': v['total_frames'],
        }
else:
    # Fallback: single video (old format)
    cap_tmp = cv2.VideoCapture(VIDEO_PATHS[0])
    video_map[0] = {
        'path': VIDEO_PATHS[0],
        'fps': cap_tmp.get(cv2.CAP_PROP_FPS),
        'width': int(cap_tmp.get(cv2.CAP_PROP_WIDTH)),
        'height': int(cap_tmp.get(cv2.CAP_PROP_HEIGHT)),
        'total_frames': int(cap_tmp.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap_tmp.release()

print(f"Labeled data: {len(data['frames'])} frames across {len(video_map)} videos")
for vid_id, info in video_map.items():
    print(f"  v{vid_id}: {os.path.basename(info['path'])} — "
          f"{info['width']}x{info['height']}, {info['fps']:.1f}fps, "
          f"{info['total_frames']} frames")

# Override stored paths with current Colab paths
for i, path in enumerate(VIDEO_PATHS):
    if i in video_map:
        video_map[i]['path'] = path

CLASSES = ['dig', 'set', 'attack', 'block', 'serve', 'idle']

# ============================================================
# COLLECT ALL LABELED DETECTIONS
# ============================================================
all_samples = []
ignored_count = 0

for frame_data in data['frames']:
    frame_idx = frame_data['frame_index']
    vid_id = frame_data.get('video_id', 0)
    for det in frame_data['detections']:
        label = det.get('label', 'idle')
        if label == 'ignore':
            ignored_count += 1
            continue
        if label not in CLASSES:
            continue

        all_samples.append({
            'video_id': vid_id,
            'frame_idx': frame_idx,
            'det_id': det['id'],
            'bbox': det['bbox'],
            'label': label,
            'confidence': det.get('confidence'),
        })

print(f"\nExcluded {ignored_count} non-player detections (labeled 'ignore')")

# Count per class (before capping)
class_counts_raw = {}
for s in all_samples:
    class_counts_raw[s['label']] = class_counts_raw.get(s['label'], 0) + 1

print(f"\nTotal samples (raw): {len(all_samples)}")
for cls in CLASSES:
    print(f"  {cls}: {class_counts_raw.get(cls, 0)}")

# ============================================================
# IDLE CAPPING — downsample idle to avoid extreme imbalance
# ============================================================
# Cap idle at MAX_IDLE_RATIO × (number of action samples)
# This keeps enough idle to learn "nothing happening" without
# drowning out the action classes.
MAX_IDLE_RATIO = 1.0   # 2x as many idles as total action samples
MIN_IDLE_COUNT = 150    # Floor — always keep at least this many

action_samples = [s for s in all_samples if s['label'] != 'idle']
idle_samples = [s for s in all_samples if s['label'] == 'idle']
num_action = len(action_samples)

idle_cap = max(MIN_IDLE_COUNT, int(num_action * MAX_IDLE_RATIO))
idle_cap = min(idle_cap, len(idle_samples))  # Don't exceed available

if len(idle_samples) > idle_cap:
    random.shuffle(idle_samples)
    idle_samples = idle_samples[:idle_cap]
    print(f"\nIdle capped: {class_counts_raw.get('idle', 0)} → {idle_cap}")

all_samples = action_samples + idle_samples

# Recount after capping
class_counts = {}
for s in all_samples:
    class_counts[s['label']] = class_counts.get(s['label'], 0) + 1

print(f"\nTotal samples (after idle cap): {len(all_samples)}")
for cls in CLASSES:
    count = class_counts.get(cls, 0)
    marker = " ⚠️  LOW — aim for 50+" if 0 < count < 40 else ""
    print(f"  {cls}: {count}{marker}")

# Warn if action classes are too small
low_classes = [c for c in CLASSES if c != 'idle' and class_counts.get(c, 0) < 40]
if low_classes:
    print(f"\n{'='*60}")
    print(f"⚠️  WARNING: These classes have too few samples for reliable training:")
    for cls in low_classes:
        current = class_counts.get(cls, 0)
        print(f"   {cls}: {current} → need ~{max(50, current)} minimum (50+ recommended)")
    print(f"\n   Go back to the labeler and specifically hunt for these actions.")
    print(f"   Tip: scrub through rallies and label only the active player,")
    print(f"   skip idle players — they're already well-represented.")
    print(f"{'='*60}")

# ============================================================
# TRAIN / VAL SPLIT (by frame to avoid data leakage)
# ============================================================
# Use (video_id, frame_idx) tuples so frames from different videos
# don't collide
unique_frames = sorted(set((s['video_id'], s['frame_idx']) for s in all_samples))
random.shuffle(unique_frames)
val_cutoff = int(len(unique_frames) * VAL_SPLIT)
val_frames = set(unique_frames[:val_cutoff])
train_frames = set(unique_frames[val_cutoff:])

train_samples = [s for s in all_samples
                 if (s['video_id'], s['frame_idx']) in train_frames]
val_samples = [s for s in all_samples
               if (s['video_id'], s['frame_idx']) in val_frames]

print(f"\nTrain: {len(train_samples)} clips from {len(train_frames)} frames")
print(f"Val:   {len(val_samples)} clips from {len(val_frames)} frames")

# Create output directories
for split in ['train', 'val']:
    for cls in CLASSES:
        os.makedirs(os.path.join(OUTPUT_DIR, split, cls), exist_ok=True)


# ============================================================
# CLIP EXTRACTION
# ============================================================
def get_padded_crop_coords(bbox, frame_w, frame_h, pad_ratio=1.5):
    """Compute a square crop region centered on bbox with padding."""
    x1, y1, x2, y2 = bbox['x1'], bbox['y1'], bbox['x2'], bbox['y2']
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    bw = x2 - x1
    bh = y2 - y1

    # Use the larger dimension to make a square, then pad
    side = max(bw, bh) * pad_ratio
    side = max(side, 80)  # Minimum crop size

    # Crop coordinates (clamped to frame)
    crop_x1 = int(max(0, cx - side / 2))
    crop_y1 = int(max(0, cy - side / 2))
    crop_x2 = int(min(frame_w, cx + side / 2))
    crop_y2 = int(min(frame_h, cy + side / 2))

    return crop_x1, crop_y1, crop_x2, crop_y2


def extract_clip_frames(cap, center_frame, half_window, num_frames,
                        vid_total, vid_w, vid_h):
    """Sample num_frames uniformly from a temporal window around center_frame."""
    start_f = max(0, center_frame - half_window)
    end_f = min(vid_total - 1, center_frame + half_window)

    # Generate uniform sample indices across the window
    if end_f - start_f + 1 <= num_frames:
        # Window smaller than needed — use all available frames
        indices = list(range(start_f, end_f + 1))
        # Pad by repeating last frame if needed
        while len(indices) < num_frames:
            indices.append(indices[-1])
    else:
        indices = np.linspace(start_f, end_f, num_frames, dtype=int).tolist()

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frames.append(frame)
        elif frames:
            frames.append(frames[-1].copy())
        else:
            frames.append(np.zeros((vid_h, vid_w, 3), dtype=np.uint8))

    return frames, indices


def save_clip_as_video(frames, crop_coords, output_path, size=224):
    """Crop each frame to the player region, resize to square, save as mp4."""
    cx1, cy1, cx2, cy2 = crop_coords

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(output_path, fourcc, 8.0, (size, size))

    for frame in frames:
        crop = frame[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            crop = np.zeros((size, size, 3), dtype=np.uint8)
        resized = cv2.resize(crop, (size, size), interpolation=cv2.INTER_LINEAR)
        writer.write(resized)

    writer.release()


def save_clip_as_npz(frames, crop_coords, output_path, size=224):
    """Save clip as .npz for faster loading during training."""
    cx1, cy1, cx2, cy2 = crop_coords
    clip_array = np.zeros((len(frames), size, size, 3), dtype=np.uint8)

    for i, frame in enumerate(frames):
        crop = frame[cy1:cy2, cx1:cx2]
        if crop.size == 0:
            crop = np.zeros((size, size, 3), dtype=np.uint8)
        clip_array[i] = cv2.resize(crop, (size, size), interpolation=cv2.INTER_LINEAR)

    np.savez_compressed(output_path, video=clip_array)


# ============================================================
# PROCESS ALL SAMPLES (grouped by video to minimize file I/O)
# ============================================================
manifest = []
processed = 0
total = len(train_samples) + len(val_samples)

# Combine all samples with their split label, then group by video
all_with_split = [(s, 'train') for s in train_samples] + \
                 [(s, 'val') for s in val_samples]

# Group by video_id for efficient processing
from collections import defaultdict
by_video = defaultdict(list)
for sample, split_name in all_with_split:
    by_video[sample['video_id']].append((sample, split_name))

for vid_id in sorted(by_video.keys()):
    vid_info = video_map[vid_id]
    vid_path = vid_info['path']
    vid_fps = vid_info['fps']
    vid_w = vid_info['width']
    vid_h = vid_info['height']
    vid_total = vid_info['total_frames']
    half_window_frames = int((CLIP_WINDOW_SEC / 2.0) * vid_fps)

    cap = cv2.VideoCapture(vid_path)
    if not cap.isOpened():
        print(f"  Cannot open video v{vid_id}: {vid_path} — skipping")
        continue

    print(f"\nProcessing v{vid_id} ({os.path.basename(vid_path)}): "
          f"{len(by_video[vid_id])} clips...")

    for sample, split_name in by_video[vid_id]:
        frame_idx = sample['frame_idx']
        det_id = sample['det_id']
        label = sample['label']
        bbox = sample['bbox']

        # Compute crop region
        crop_coords = get_padded_crop_coords(bbox, vid_w, vid_h, CROP_PAD_RATIO)

        # Extract temporal window of frames from THIS video
        clip_frames, frame_indices = extract_clip_frames(
            cap, frame_idx, half_window_frames, NUM_FRAMES,
            vid_total, vid_w, vid_h
        )

        # Save — include video ID in clip name
        clip_name = f"v{vid_id}_clip_f{frame_idx:06d}_d{det_id}"
        npz_path = os.path.join(OUTPUT_DIR, split_name, label, clip_name + ".npz")
        mp4_path = os.path.join(OUTPUT_DIR, split_name, label, clip_name + ".mp4")

        save_clip_as_npz(clip_frames, crop_coords, npz_path, CROP_SIZE)
        save_clip_as_video(clip_frames, crop_coords, mp4_path, CROP_SIZE)

        manifest.append({
            'split': split_name,
            'label': label,
            'clip_name': clip_name,
            'video_id': vid_id,
            'source_frame': frame_idx,
            'det_id': det_id,
            'bbox': bbox,
            'crop_coords': list(crop_coords),
            'frame_indices': frame_indices,
            'npz_path': npz_path,
            'mp4_path': mp4_path,
        })

        processed += 1
        if processed % 50 == 0:
            print(f"  [{processed}/{total}] Extracted {clip_name} -> {label}")

    cap.release()

# Save manifest
manifest_path = os.path.join(OUTPUT_DIR, "clip_manifest.json")
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"\nDone! {processed} clips extracted.")
print(f"Manifest: {manifest_path}")

# Summary
for split_name in ['train', 'val']:
    split_clips = [m for m in manifest if m['split'] == split_name]
    print(f"\n{split_name}:")
    for cls in CLASSES:
        count = len([m for m in split_clips if m['label'] == cls])
        if count > 0:
            print(f"  {cls}: {count}")



Labeled data: 1134 frames across 5 videos
  v0: clip_01_19_56_to_01_20_17_trimmed.mp4 — 1920x1080, 62.1fps, 1118 frames
  v1: clip_01_37_16_to_01_37_32_clipped.mp4 — 1920x1080, 62.0fps, 975 frames
  v2: clip_01_44_20_to_01_44_57_clipped.mp4 — 1920x1080, 60.0fps, 2188 frames
  v3: clip_01_44_51_to_01_45_04.mp4 — 1920x1080, 62.1fps, 803 frames
  v4: clip_42_42_to_43_06_clipped.MP4 — 1920x1080, 62.0fps, 1297 frames

Excluded 859 non-player detections (labeled 'ignore')

Total samples (raw): 26126
  dig: 89
  set: 78
  attack: 71
  block: 184
  serve: 103
  idle: 25601

Idle capped: 25601 → 525

Total samples (after idle cap): 1050
  dig: 89
  set: 78
  attack: 71
  block: 184
  serve: 103
  idle: 525

Train: 896 clips from 590 frames
Val:   154 clips from 104 frames

Processing v0 (clip_01_19_56_to_01_20_17_trimmed.mp4): 172 clips...
  [50/1050] Extracted v0_clip_f000680_d10 -> dig


KeyboardInterrupt: 